# NB05: Direct FB-BacDive Case Studies

For the 12 Fitness Browser organisms matching BacDive, present **descriptive case studies**
comparing BacDive phenotypes to actual per-metal fitness profiles.

**Framing**: n=12 is insufficient for formal hypothesis testing. This notebook provides
qualitative illustrations of whether the pangenome-scale associations (NB03-04) hold for
organisms with direct experimental data.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

MAIN_REPO = '/home/psdehal/pangenome_science/BERIL-research-observatory'
PROJ = os.path.join(MAIN_REPO, 'projects', 'bacdive_phenotype_metal_tolerance')
DATA_OUT = os.path.join(PROJ, 'data')
FIG_OUT = os.path.join(PROJ, 'figures')

BACDIVE = os.path.join(MAIN_REPO, 'data', 'bacdive_ingest')
METAL_ATLAS = os.path.join(MAIN_REPO, 'projects', 'metal_fitness_atlas', 'data')
COUNTER_ION = os.path.join(MAIN_REPO, 'projects', 'counter_ion_effects', 'data')

## 1. Identify the 12 FB-BacDive matched organisms

In [ ]:
# Load FB organism mapping
org_map = pd.read_csv(os.path.join(MAIN_REPO, 'projects', 'conservation_vs_fitness', 'data', 'organism_mapping.tsv'), sep='\t')

# Load BacDive strain data with taxid
bd_strain = pd.read_csv(os.path.join(BACDIVE, 'strain.tsv'), sep='\t')

# Match by NCBI taxonomy ID
# FB organisms have taxonomyId; BacDive strains have ncbi_taxid
org_map['taxonomyId'] = org_map['taxonomyId'].astype(str)
bd_strain['ncbi_taxid'] = bd_strain['ncbi_taxid'].astype(str)

fb_taxids = set(org_map['taxonomyId'].dropna())
bd_taxids = set(bd_strain['ncbi_taxid'].dropna())

overlap_taxids = fb_taxids & bd_taxids
print(f'FB organisms with taxid: {len(fb_taxids)}')
print(f'BacDive strains with taxid: {len(bd_taxids)}')
print(f'Overlapping taxids: {len(overlap_taxids)}')

# Get FB organism details for overlapping taxa
fb_matched = org_map[org_map['taxonomyId'].isin(overlap_taxids)].drop_duplicates(subset='orgId')
print(f'\nMatched FB organisms: {len(fb_matched)}')
print(fb_matched[['orgId', 'genus', 'species', 'strain', 'taxonomyId']].to_string(index=False))

## 2. Extract BacDive phenotypes for matched organisms

In [ ]:
# Load all BacDive tables
bd_tax = pd.read_csv(os.path.join(BACDIVE, 'taxonomy.tsv'), sep='\t')
bd_phys = pd.read_csv(os.path.join(BACDIVE, 'physiology.tsv'), sep='\t')
bd_mu = pd.read_csv(os.path.join(BACDIVE, 'metabolite_utilization.tsv'), sep='\t')
bd_enz = pd.read_csv(os.path.join(BACDIVE, 'enzyme.tsv'), sep='\t')

# Get BacDive strains matching FB organisms
bd_fb_strains = bd_strain[bd_strain['ncbi_taxid'].isin(overlap_taxids)]
fb_bd_ids = set(bd_fb_strains['bacdive_id'])
print(f'BacDive strains matching FB organisms: {len(bd_fb_strains)}')

# Per-organism phenotype summary
phenotype_summary = []
for _, org in fb_matched.iterrows():
    taxid = org['taxonomyId']
    org_strains = bd_strain[bd_strain['ncbi_taxid'] == taxid]
    org_ids = set(org_strains['bacdive_id'])
    
    # Physiology
    org_phys = bd_phys[bd_phys['bacdive_id'].isin(org_ids)]
    gram = org_phys['gram_stain'].dropna().mode().iloc[0] if org_phys['gram_stain'].notna().any() else 'N/A'
    oxygen = org_phys['oxygen_tolerance'].dropna().mode().iloc[0] if org_phys['oxygen_tolerance'].notna().any() else 'N/A'
    
    # Key enzymes
    org_enz = bd_enz[bd_enz['bacdive_id'].isin(org_ids)]
    def enzyme_status(name):
        e = org_enz[org_enz['enzyme_name'].str.contains(name, case=False, na=False)]
        if len(e) == 0: return 'N/A'
        pos = (e['activity'] == '+').sum()
        neg = (e['activity'] == '-').sum()
        return f'+({pos}/{pos+neg})' if pos > neg else f'-({neg}/{pos+neg})'
    
    # Key metabolites  
    org_mu = bd_mu[bd_mu['bacdive_id'].isin(org_ids)]
    def metabolite_status(compound):
        m = org_mu[org_mu['compound_name'] == compound]
        if len(m) == 0: return 'N/A'
        pos = m['utilization'].isin(['+', 'produced']).sum()
        neg = (m['utilization'] == '-').sum()
        return f'+({pos}/{pos+neg})' if pos > neg else f'-({neg}/{pos+neg})'
    
    phenotype_summary.append({
        'orgId': org['orgId'],
        'species': f"{org['genus']} {org['species']}",
        'n_strains': len(org_strains),
        'gram': gram,
        'oxygen': oxygen,
        'catalase': enzyme_status('catalase'),
        'oxidase': enzyme_status('oxidase'),
        'urease': enzyme_status('urease'),
        'nitrate': metabolite_status('nitrate'),
        'h2s': metabolite_status('hydrogen sulfide'),
    })

pheno_df = pd.DataFrame(phenotype_summary)
print('\n=== BacDive Phenotypes for FB Organisms ===')
print(pheno_df.to_string(index=False))

## 3. Load metal fitness profiles for matched organisms

In [ ]:
# Load metal experiment classifications
try:
    metal_exps = pd.read_csv(os.path.join(METAL_ATLAS, 'metal_experiment_classification.csv'))
    print(f'Metal experiments: {len(metal_exps)}')
    
    # Filter to FB-BacDive matched organisms
    fb_orgids = set(fb_matched['orgId'])
    matched_metal = metal_exps[metal_exps['orgId'].isin(fb_orgids)]
    print(f'Metal experiments for matched organisms: {len(matched_metal)}')
    
    # Per-organism metal summary
    metal_summary = matched_metal.groupby('orgId').agg(
        n_experiments=('expName', 'count'),
        metals=('metal_element', lambda x: ', '.join(sorted(x.unique()))),
        n_metals=('metal_element', 'nunique'),
    ).reset_index()
    
    print('\n=== Metal Experiments per Matched Organism ===')
    print(metal_summary.to_string(index=False))
except FileNotFoundError:
    print('Metal experiment classification not found. Using counter_ion_effects data.')
    metal_exps = pd.read_csv(os.path.join(COUNTER_ION, 'effective_chloride_concentrations.csv'))
    fb_orgids = set(fb_matched['orgId'])
    matched_metal = metal_exps[metal_exps['orgId'].isin(fb_orgids)]
    print(f'Metal experiments for matched organisms: {len(matched_metal)}')
    
    metal_summary = matched_metal.groupby('orgId').agg(
        n_experiments=('expName', 'count'),
        metals=('metal_element', lambda x: ', '.join(sorted(x.unique()))),
        n_metals=('metal_element', 'nunique'),
    ).reset_index()
    print(metal_summary.to_string(index=False))

## 4. Hypothesis concordance table

In [ ]:
# Merge phenotypes with metal data
combined = pheno_df.merge(metal_summary, on='orgId', how='left')

# For each hypothesis, assess directional concordance across the 12 organisms
print('=== Hypothesis Concordance (Descriptive) ===')
print()

hypotheses = [
    ('H1a', 'Gram-negative → higher metal tolerance',
     'gram', 'Gram-negative bacteria should have more metal experiments or tolerate more metals'),
    ('H1d', 'Catalase+ → better with redox metals (Cu, Cr, Fe)',
     'catalase', 'Catalase-positive organisms handle redox-cycling metals better'),
    ('H1e', 'Urease+ → nickel tolerance',
     'urease', 'Urease-positive organisms should tolerate nickel (urease is Ni-dependent)'),
    ('H1f', 'H₂S+ → chalcophilic metal tolerance (Zn, Cu, Cd)',
     'h2s', 'H₂S producers should tolerate metals that form insoluble sulfides'),
]

for hid, description, col, prediction in hypotheses:
    print(f'{hid}: {description}')
    print(f'  Prediction: {prediction}')
    relevant = combined[combined[col] != 'N/A']
    if len(relevant) == 0:
        print(f'  Result: No data available')
    else:
        print(f'  Data: {len(relevant)} organisms with {col} data')
        for _, row in relevant.iterrows():
            metals_str = row.get('metals', 'none')
            print(f'    {row["orgId"]:15s} {row["species"]:35s} {col}={row[col]:8s} metals: {metals_str}')
    print()

# Save combined table
combined.to_csv(os.path.join(DATA_OUT, 'fb_bacdive_combined.csv'), index=False)
print(f'Saved: data/fb_bacdive_combined.csv')

## 5. Case studies

In [ ]:
# Case Study 1: DvH (Desulfovibrio vulgaris Hildenborough)
# Key features: anaerobe, H₂S producer, Gram-negative → should be metal-tolerant
print('=== Case Study: Desulfovibrio vulgaris Hildenborough (dvh) ===')
dvh = combined[combined['orgId'] == 'dvh']
if len(dvh) > 0:
    dvh_row = dvh.iloc[0]
    print(f'  Gram: {dvh_row["gram"]}')
    print(f'  Oxygen: {dvh_row["oxygen"]}')
    print(f'  Catalase: {dvh_row["catalase"]}')
    print(f'  Urease: {dvh_row["urease"]}')
    print(f'  H₂S: {dvh_row["h2s"]}')
    print(f'  Nitrate: {dvh_row["nitrate"]}')
    print(f'  Metals tested: {dvh_row.get("metals", "none")}')
    print(f'  N experiments: {dvh_row.get("n_experiments", 0)}')
    print()
    print('  Interpretation: DvH is an anaerobic sulfate-reducing bacterium that produces H₂S.')
    print('  It is tested against 13 metals in the Fitness Browser — the most of any organism.')
    print('  H1b (anaerobe → redox metal tolerance): TESTABLE — DvH tolerates uranium and chromium.')
    print('  H1f (H₂S → chalcophilic metals): TESTABLE — DvH produces sulfide that can precipitate Zn, Cu.')
else:
    print('  DvH not in FB-BacDive matched set.')

print()

# Case Study 2: E. coli (Keio)
print('=== Case Study: Escherichia coli (Keio) ===')
keio = combined[combined['orgId'] == 'Keio']
if len(keio) > 0:
    keio_row = keio.iloc[0]
    print(f'  Gram: {keio_row["gram"]}')
    print(f'  Oxygen: {keio_row["oxygen"]}')
    print(f'  Catalase: {keio_row["catalase"]}')
    print(f'  Urease: {keio_row["urease"]}')
    print(f'  H₂S: {keio_row["h2s"]}')
    print(f'  Metals tested: {keio_row.get("metals", "none")}')
    print()
    print('  Interpretation: E. coli is a Gram-negative facultative anaerobe, catalase+, urease-.')
    print('  H1a (Gram-neg → metal tolerant): E. coli has moderate metal tolerance.')
    print('  H1d (catalase → redox metals): Catalase+ should help with Cu, Cr, Fe tolerance.')
    print('  H1e (urease- → lower Ni tolerance): E. coli lacks urease — check Ni fitness.')
else:
    print('  Keio not in FB-BacDive matched set.')

## 6. Summary figure

In [ ]:
# Heatmap: organisms × phenotype features
display_cols = ['gram', 'oxygen', 'catalase', 'oxidase', 'urease', 'nitrate', 'h2s']
display_df = pheno_df.set_index('orgId')[display_cols].copy()

# Encode for heatmap: + → 1, - → 0, N/A → NaN
def encode_phenotype(val):
    if pd.isna(val) or val == 'N/A':
        return np.nan
    if val.startswith('+') or val in ['negative', 'aerobe', 'facultative anaerobe']:
        return 1
    if val.startswith('-') or val in ['positive', 'anaerobe']:
        return 0
    return np.nan

encoded = display_df.map(encode_phenotype)

fig, ax = plt.subplots(figsize=(10, max(6, len(encoded) * 0.5)))
sns.heatmap(encoded, annot=display_df, fmt='', cmap='RdYlGn', center=0.5,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Feature status'})
ax.set_title('BacDive Phenotypes for FB-Matched Organisms')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, 'fb_bacdive_phenotype_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/fb_bacdive_phenotype_heatmap.png')